# Context table: logistic regression, a small MLP and the probe on the same 8 inputs

The plan's context table. For each 8-input set it reports what simple classical models reach on exactly those inputs, next to the circuit's exact twin and the linear probe on all features. The honesty rule asks for this table: the gains are about input design under a qubit budget, not quantum effects.

**Rows (8-input sets), per backbone:**
- the five baselines of `search_analysis.ipynb`: first 8 PCA, strongest 8, PLS 8, MI top 8 and greedy forward selection with the search fitness, each with the twin feature set that scores higher;
- each search method's top genome: the single genome with the highest fitness over all of the method's runs, never picked by val;
- on L/14, the top genome of the held-out ablation (GA + held-out term), when its output is attached;
- or, with `use_finalists = True`, the genomes listed in a `finalists.csv` input (columns `backbone`, `label`, `genome`), plus PLS 8 for reference.

**Models on each row's 8 inputs:**
- **twin:** logistic regression on the circuit's sin/cos features (cos only for RX or H+RZ encoding with an RY rotation), the function class of the additive circuit;
- **logistic regression:** on the 8 raw pool scores;
- **MLP:** one hidden layer of 4 units on the 8 raw scores (8 x 4 + 4 + 4 + 1 = 41 weights; the circuit with one Rot layer has 33);
- **linear probe:** on all features of the backbone, read from `pool_prep`'s bottleneck table, which uses the same protocol.

**Protocol, the same as `pool_prep`'s bottleneck table:** every model picks its setting (C for logistic regression, alpha for the MLP) by video AUC on the inner fitness frames after training on search-train, then is refit on every training frame it may use. Scores are video AUC (a video's score is the mean of its frame probabilities).
- **val:** trained on all of FF++ train, scored on all of FF++ val, with a 95% bootstrap interval over val videos.
- **held-out:** for each manipulation, trained without its fakes and scored on val real videos plus that manipulation's fakes. PLS (16 components, `scale=True`) and every column's angle range are refit inside the fold, so nothing that builds the inputs has seen the held-out fakes. PCA uses no labels and stays fixed.

FF++ test and Celeb-DF are not touched.

**Inputs (Add Input):**
- `malakkhaled202300369/nic-pool` (folder `pool/`)
- the output of `faridaali202301940/genome-search` (folder `genome_search/`)
- the output of `faridaali202301940/genome-search-ablation` (folder `genome_search_ablation/`), optional
- raw features: `malakkhaled202300369/nic-clip768-l14-dataset` (L/14), `malakkhaled202300369/nic-clip-dataset` (B/16), `malakkhaled202300369/efficientnet-features` (EfficientNet-B4)

**Settings:** Accelerator None (CPU). No installs needed.

## 1. Settings

In [1]:
backbones = ["l14", "b16", "effb4"]
methods = ["random", "GA", "PSO", "memetic GA", "DE", "GWO"]
use_ablation = True          # add the held-out ablation's top genome on L/14 when its output is attached
use_finalists = False        # True: rows come from a finalists.csv input instead of the search
c_grid = [0.001, 0.01, 0.1, 1.0]      # logistic regression and twin, picked on the inner split as in pool_prep
mlp_hidden = 4                        # 8 x 4 + 4 + 4 + 1 = 41 weights
alpha_grid = [1e-4, 1e-3, 1e-2]       # MLP weight decay, picked on the inner split
twin_c = 0.01                # the search fitness, used only to build the greedy baseline
fit_per_class = 1000         # the search's fitness frames, used only for the greedy baseline
n_pls = 16
low, high = 0.5, 99.5        # angle range percentiles, as in pool_prep
n_boot = 1000
n_jobs = 0                   # 0 = one job per CPU core
machine_name = "kaggle-farida"
seed = 0
out_dir = "/kaggle/working/context_table"

# raw features per backbone: (folder, file suffix), the same lookup as pool_prep
raw_features = {"b16": ("features", "_clip.npy"), "l14": ("features_l14", "_clip.npy"), "effb4": ("features", "_effb4.npy")}

## 2. Imports

In [2]:
import os
# one thread per process, set before numpy loads
os.environ["OMP_NUM_THREADS"] = "1"
import glob
import json
import time
import warnings
import platform
import multiprocessing as mp
import numpy as np
import pandas as pd
import sklearn
from sklearn.cross_decomposition import PLSRegression
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore", category=ConvergenceWarning)
os.makedirs(out_dir, exist_ok=True)
if n_jobs == 0:
    n_jobs = os.cpu_count()
start_all = time.time()
print("backbones", backbones, "| jobs", n_jobs, "| finalists", use_finalists)

backbones ['l14', 'b16', 'effb4'] | jobs 4 | finalists False


## 3. Inputs

Several attached datasets hold files with the same name, so every file is looked up by name and by the folder it sits in.

In [3]:
def find_all(filename):
    return sorted(glob.glob("/kaggle/input/**/" + filename, recursive=True))


def find_one(filename, folder=None):
    hits = find_all(filename)
    if folder is not None:
        hits = [p for p in hits if os.path.basename(os.path.dirname(p)) == folder]
    assert len(hits) == 1, (filename, folder, hits)
    return hits[0]


pool_dir = os.path.dirname(find_one("columns.json", "pool"))
bottleneck = pd.read_csv(f"{pool_dir}/bottleneck_table.csv")
search_log = pd.read_csv(find_one("search_log.csv", "genome_search"))
print("search log:", len(search_log), "evaluations")

ablation_log = None
if use_ablation:
    hits = [p for p in find_all("search_log.csv") if os.path.basename(os.path.dirname(p)) == "genome_search_ablation"]
    if len(hits) == 1:
        ablation_log = pd.read_csv(hits[0])
        print("ablation log:", len(ablation_log), "evaluations")
    else:
        print("ablation output not attached, its row is skipped")

finalists = None
if use_finalists:
    finalists = pd.read_csv(find_one("finalists.csv"))
    print("finalists:", len(finalists))

search log: 150000 evaluations
ablation log: 10000 evaluations


## 4. Loading a backbone

`load_backbone` fills `D` with everything the models need for one backbone, then builds the inputs of every fold:
- `all`: PCA scores as saved, PLS refit on all search-train frames (this reproduces the saved pool), angle range from all train frames;
- one fold per manipulation: PLS refit on the search-train frames without that manipulation's fakes, angle range from the kept train frames.

Each fold holds the raw scores (PCA, then PLS) and the angles, for train and val. The worker processes start after this, so they share it.

In [4]:
D = {}


def video_ids(rows):
    return rows.groupby(["method", "video_id"]).ngroup().to_numpy()


def load_backbone(b):
    D.clear()
    rows = pd.read_csv(f"{pool_dir}/ffpp_train_rows.csv", dtype={"video_id": str})
    val_rows = pd.read_csv(f"{pool_dir}/ffpp_val_rows.csv", dtype={"video_id": str})
    is_inner = np.load(f"{pool_dir}/ffpp_train_is_inner.npy").astype(bool)
    columns = json.load(open(f"{pool_dir}/columns.json"))
    pool = np.load(f"{pool_dir}/{b}_ffpp_train_pool.npy").astype(np.float64)
    scores_tr = np.load(f"{pool_dir}/{b}_ffpp_train_scores.npy").astype(np.float64)
    scores_va = np.load(f"{pool_dir}/{b}_ffpp_val_scores.npy").astype(np.float64)
    n_pca = sum(c.startswith("pca") for c in columns)
    y_tr = rows["label"].to_numpy()
    y_va = val_rows["label"].to_numpy()
    m_tr = rows["method"].to_numpy()
    m_va = val_rows["method"].to_numpy()
    search = ~is_inner
    fake_methods = sorted(set(m_tr[y_tr == 1]))

    folder, suffix = raw_features[b]
    xn = {}
    for split, r in [("ffpp_train", rows), ("ffpp_val", val_rows)]:
        path = find_one(split + suffix, folder)
        raw_rows = pd.read_csv(os.path.join(os.path.dirname(path), split + "_rows.csv"), dtype={"video_id": str})
        assert (raw_rows["path"].to_numpy() == r["path"].to_numpy()).all(), (b, split, "raw features not in pool row order")
        x = np.load(path).astype(np.float64)
        xn[split] = x / np.linalg.norm(x, axis=1, keepdims=True)
        del x

    folds = {}
    keep_all = np.ones(len(y_tr), dtype=bool)
    fold_keeps = [("all", keep_all, np.ones(len(y_va), dtype=bool))]
    for m in fake_methods:
        fold_keeps.append((m, (y_tr == 0) | (m_tr != m), (y_va == 0) | (m_va == m)))
    for fold, keep, val_keep in fold_keeps:
        pls = PLSRegression(n_components=n_pls, scale=True).fit(xn["ffpp_train"][search & keep], y_tr[search & keep])
        s_tr = np.hstack([scores_tr[:, :n_pca], pls.transform(xn["ffpp_train"])])
        s_va = np.hstack([scores_va[:, :n_pca], pls.transform(xn["ffpp_val"])])
        lo = np.percentile(s_tr[keep], low, axis=0)
        hi = np.percentile(s_tr[keep], high, axis=0)
        folds[fold] = {"keep": keep, "val_keep": val_keep, "s_tr": s_tr, "s_va": s_va,
                       "a_tr": np.clip((s_tr - lo) / (hi - lo), 0.0, 1.0) * np.pi,
                       "a_va": np.clip((s_va - lo) / (hi - lo), 0.0, 1.0) * np.pi}
    del xn
    diff = np.abs(folds["all"]["a_tr"] - pool).max()
    print(f"{b}: all-fold angles against the saved pool, largest difference {diff:.2e}")
    assert diff < 1e-3, "the attached raw features do not reproduce the saved pool"

    # the search's fitness frames, for the greedy baseline only (same code and seed as the search)
    st_index = np.where(search)[0]
    st_rows = rows.iloc[st_index].copy()
    st_rows["row"] = st_index
    rng = np.random.default_rng(0)
    real_pick = st_rows[st_rows["label"] == 0].groupby("video_id").sample(n=2, random_state=0)["row"].to_numpy()
    fake_pick = st_rows[st_rows["label"] == 1].groupby(["method", "video_id"]).sample(n=1, random_state=0)["row"].to_numpy()
    fit_index = np.concatenate([rng.choice(real_pick, fit_per_class, replace=False),
                                rng.choice(fake_pick, fit_per_class, replace=False)])

    vid_va = video_ids(val_rows)
    n_videos = vid_va.max() + 1
    video_y = np.zeros(n_videos)
    video_y[vid_va] = y_va
    D.update(backbone=b, columns=columns, n_cols=len(columns), n_pca=n_pca, pool=pool, y_tr=y_tr, y_va=y_va,
             search=search, inner=is_inner, fake_methods=fake_methods, folds=folds, fit_index=fit_index,
             inner_index=np.where(is_inner)[0], vid_tr=video_ids(rows), vid_va=vid_va, video_y=video_y,
             video_count=np.bincount(vid_va, minlength=n_videos),
             boot=np.random.default_rng(seed).integers(0, n_videos, (n_boot, n_videos)))
    print(f"{b}: {len(columns)} columns, folds {list(folds)}, {n_videos} val videos")

## 5. Genomes and the baselines

The genome format and twin rule are the search's. The baselines are built exactly as in `search_analysis.ipynb`: fixed masks scored with both twin feature sets on the search fitness, and greedy forward selection run once per feature set; each keeps the feature set with the higher fitness.

In [5]:
twin_sets = {"sin, cos": ("RY", "Rot"), "cos": ("RX", "RY")}


def twin_features(x, g):
    if g["encoding"] != "RY" and g["rotation"] == "RY":
        return np.cos(x)
    return np.hstack([np.sin(x), np.cos(x)])


def twin_set(g):
    return "cos" if g["encoding"] != "RY" and g["rotation"] == "RY" else "sin, cos"


def key_of(g):
    return ".".join(D["columns"][p] for p in g["picks"]) + "|" + g["encoding"] + "|" + g["rotation"]


def with_set(picks, name):
    encoding, rotation = twin_sets[name]
    return {"picks": sorted(int(p) for p in picks), "encoding": encoding, "rotation": rotation}


def search_fitness(g):
    fi, ii, y = D["fit_index"], D["inner_index"], D["y_tr"]
    x = D["pool"][:, g["picks"]]
    model = make_pipeline(StandardScaler(), LogisticRegression(C=twin_c, max_iter=5000))
    model.fit(twin_features(x[fi], g), y[fi])
    return roc_auc_score(y[ii], model.predict_proba(twin_features(x[ii], g))[:, 1])


def greedy(name, workers):
    picks = []
    for step in range(8):
        candidates = [j for j in range(D["n_cols"]) if j not in picks]
        scores = workers.map(search_fitness, [with_set(picks + [j], name) for j in candidates])
        picks.append(candidates[int(np.argmax(scores))])
    return picks


def baseline_rows(workers):
    y, search, pool = D["y_tr"], D["search"], D["pool"]
    strength = []
    for j in range(D["n_cols"]):
        auc = roc_auc_score(y[search], pool[search, j])
        strength.append(max(auc, 1 - auc))
    mi = mutual_info_classif(pool[search], y[search], random_state=seed)
    masks = {
        "first 8 PCA": list(range(8)),
        "strongest 8": np.argsort(strength)[::-1][:8].tolist(),
        "PLS 8": list(range(D["n_pca"], D["n_pca"] + 8)),
        "MI top 8": np.argsort(mi)[::-1][:8].tolist(),
    }
    out = []
    for label, picks in masks.items():
        best = max((with_set(picks, n) for n in twin_sets), key=search_fitness)
        out.append({"row": label, "source": "baseline", "genome": best, "fitness": search_fitness(best)})
    best = max((with_set(greedy(n, workers), n) for n in twin_sets), key=search_fitness)
    out.append({"row": "greedy", "source": "baseline", "genome": best, "fitness": search_fitness(best)})
    return out


def top_genome(log, method_name):
    r = log[(log["backbone"] == D["backbone"]) & (log["method"] == method_name)]
    best = r.loc[r["fitness"].idxmax()]
    g = json.loads(best["genome"])
    assert key_of(g) == best["key"], "the search log does not match this pool's columns"
    return {"row": method_name + " top", "source": "search", "genome": g, "fitness": float(best["fitness"])}


def rows_for_backbone(workers):
    if use_finalists:
        f = finalists[finalists["backbone"] == D["backbone"]]
        out = [{"row": r["label"], "source": "finalist", "genome": json.loads(r["genome"]), "fitness": np.nan}
               for _, r in f.iterrows()]
        pls8 = with_set(range(D["n_pca"], D["n_pca"] + 8), "cos")
        out.append({"row": "PLS 8", "source": "baseline", "genome": pls8, "fitness": search_fitness(pls8)})
        return out
    out = baseline_rows(workers)
    for m in methods:
        out.append(top_genome(search_log, m))
    if ablation_log is not None and D["backbone"] in set(ablation_log["backbone"]):
        out.append(top_genome(ablation_log, "GA + held-out"))
        out[-1]["source"] = "ablation"
    return out

## 6. Models and scoring

`fit_tuned` is `pool_prep`'s rule: try each setting trained on the kept search-train frames, keep the one with the best video AUC on the kept inner frames, refit on every kept training frame and predict val. A task scores one row on one fold with all three models; the `all` fold also gets bootstrap intervals.

In [6]:
def make_model(kind, setting):
    if kind == "MLP":
        return make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(mlp_hidden,), alpha=setting,
                                                             max_iter=500, early_stopping=True, random_state=seed))
    return make_pipeline(StandardScaler(), LogisticRegression(C=setting, max_iter=5000))


def train_video_auc(prob, mask):
    df = pd.DataFrame({"v": D["vid_tr"][mask], "p": prob, "y": D["y_tr"][mask]})
    g = df.groupby("v").agg(p=("p", "mean"), y=("y", "first"))
    return roc_auc_score(g["y"], g["p"])


def val_video_scores(prob):
    return np.bincount(D["vid_va"], weights=prob, minlength=len(D["video_y"])) / D["video_count"]


def fit_tuned(kind, x_tr, x_va, fold):
    keep = D["folds"][fold]["keep"]
    fit_rows, tune_rows = keep & D["search"], keep & D["inner"]
    grid = alpha_grid if kind == "MLP" else c_grid
    best = None
    for setting in grid:
        model = make_model(kind, setting).fit(x_tr[fit_rows], D["y_tr"][fit_rows])
        auc = train_video_auc(model.predict_proba(x_tr[tune_rows])[:, 1], tune_rows)
        if best is None or auc > best[1]:
            best = (setting, auc)
    model = make_model(kind, best[0]).fit(x_tr[keep], D["y_tr"][keep])
    return best[0], model.predict_proba(x_va)[:, 1]


def score_task(task):
    i, fold = task
    g = ROWS[i]["genome"]
    F = D["folds"][fold]
    picks = g["picks"]
    inputs = {
        "twin": (twin_features(F["a_tr"][:, picks], g), twin_features(F["a_va"][:, picks], g)),
        "logistic regression": (F["s_tr"][:, picks], F["s_va"][:, picks]),
        "MLP": (F["s_tr"][:, picks], F["s_va"][:, picks]),
    }
    out = []
    for kind, (x_tr, x_va) in inputs.items():
        setting, prob = fit_tuned("MLP" if kind == "MLP" else "lr", x_tr, x_va, fold)
        scores = val_video_scores(prob)
        if fold == "all":
            auc = roc_auc_score(D["video_y"], scores)
            boot = [roc_auc_score(D["video_y"][b], scores[b]) for b in D["boot"]]
            row = {"val": auc, "val low": np.percentile(boot, 2.5), "val high": np.percentile(boot, 97.5)}
        else:
            vk = np.zeros(len(D["video_y"]), dtype=bool)
            vk[D["vid_va"][F["val_keep"]]] = True
            row = {"held-out " + fold: roc_auc_score(D["video_y"][vk], scores[vk])}
        out.append(dict(row, i=i, fold=fold, model=kind, setting=setting))
    return out


def n_weights(kind, g):
    if kind == "twin":
        return (8 if twin_set(g) == "cos" else 16) + 1
    if kind == "MLP":
        return 8 * mlp_hidden + mlp_hidden + mlp_hidden + 1
    return 9

## 7. Run everything

For each backbone: build the folds, the rows, then score every row on every fold with the three models in parallel.

In [7]:
ROWS = []
results = []
for b in backbones:
    t0 = time.time()
    load_backbone(b)
    with mp.get_context("fork").Pool(n_jobs) as workers:
        ROWS = rows_for_backbone(workers)
    print(b, "rows:", [r["row"] for r in ROWS], "(%.0f s)" % (time.time() - t0))
    tasks = [(i, fold) for i in range(len(ROWS)) for fold in D["folds"]]
    with mp.get_context("fork").Pool(n_jobs) as workers:
        parts = workers.map(score_task, tasks)
    merged = {}
    for part in parts:
        for r in part:
            key = (r["i"], r["model"])
            entry = merged.setdefault(key, {})
            if r["fold"] == "all":
                entry["setting"] = r["setting"]
            entry.update({k: v for k, v in r.items() if k not in ("i", "fold", "model", "setting")})
    for (i, kind), entry in merged.items():
        g = ROWS[i]["genome"]
        held = [entry["held-out " + m] for m in D["fake_methods"]]
        results.append(dict({"backbone": b, "row": ROWS[i]["row"], "source": ROWS[i]["source"], "model": kind,
                             "weights": n_weights(kind, g), "fitness": ROWS[i]["fitness"], "twin set": twin_set(g),
                             "key": key_of(g), "genome": json.dumps(g)}, **entry,
                            **{"held-out mean": float(np.mean(held)), "held-out worst": float(np.min(held))}))
    print(b, "scored %d rows x %d folds x 3 models in %.1f min" % (len(ROWS), len(D["folds"]), (time.time() - t0) / 60))

table = pd.DataFrame(results)

# the linear probe on all features, from pool_prep's bottleneck table (same protocol)
fake_methods = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]
probe = bottleneck[bottleneck["input"] == "full linear probe"]
for _, r in probe.iterrows():
    if r["backbone"] not in backbones:
        continue
    dims = {"b16": 512, "l14": 768, "effb4": 1792}[r["backbone"]]
    table = pd.concat([table, pd.DataFrame([dict({"backbone": r["backbone"], "row": "all features", "source": "pool_prep",
                                                  "model": "linear probe", "weights": dims + 1, "setting": r["C"],
                                                  "val": r["val"], "held-out mean": r["held-out mean"],
                                                  "held-out worst": r["held-out worst"]},
                                                 **{"held-out " + m: r["held-out " + m] for m in fake_methods})])],
                      ignore_index=True)
table.to_csv(f"{out_dir}/context_table.csv", index=False)
print("done in %.1f min" % ((time.time() - start_all) / 60))

l14: all-fold angles against the saved pool, largest difference 2.58e-07
l14: 144 columns, folds ['all', 'Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures'], 700 val videos
l14 rows: ['first 8 PCA', 'strongest 8', 'PLS 8', 'MI top 8', 'greedy', 'random top', 'GA top', 'PSO top', 'memetic GA top', 'DE top', 'GWO top', 'GA + held-out top'] (67 s)
l14 scored 12 rows x 5 folds x 3 models in 5.2 min
b16: all-fold angles against the saved pool, largest difference 2.52e-07
b16: 144 columns, folds ['all', 'Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures'], 700 val videos
b16 rows: ['first 8 PCA', 'strongest 8', 'PLS 8', 'MI top 8', 'greedy', 'random top', 'GA top', 'PSO top', 'memetic GA top', 'DE top', 'GWO top'] (61 s)
b16 scored 11 rows x 5 folds x 3 models in 4.5 min
effb4: all-fold angles against the saved pool, largest difference 2.61e-07
effb4: 144 columns, folds ['all', 'Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures'], 700 val videos
effb4 rows: ['first 8 PCA', 'stronge

## 8. Check against pool_prep

Logistic regression on the raw PLS 8 and first-8-PCA scores follows the same protocol as `pool_prep`'s "raw PLS 8" and "raw first 8 PCA" rows, so the numbers should agree. A mismatch means the folds or the tuning differ.

In [8]:
check = []
for b in backbones:
    for mine, theirs in [("PLS 8", "raw PLS 8"), ("first 8 PCA", "raw first 8 PCA")]:
        a = table[(table["backbone"] == b) & (table["row"] == mine) & (table["model"] == "logistic regression")]
        t = bottleneck[(bottleneck["backbone"] == b) & (bottleneck["input"] == theirs)]
        if len(a) and len(t):
            check.append({"backbone": b, "input": mine, "val here": a["val"].iloc[0], "val pool_prep": t["val"].iloc[0],
                          "held-out mean here": a["held-out mean"].iloc[0],
                          "held-out mean pool_prep": t["held-out mean"].iloc[0]})
print(pd.DataFrame(check).round(4).to_string(index=False))

backbone       input  val here  val pool_prep  held-out mean here  held-out mean pool_prep
     l14       PLS 8    0.8912         0.8912              0.8146                   0.8146
     l14 first 8 PCA    0.7393         0.7393              0.7375                   0.7375
     b16       PLS 8    0.8500         0.8500              0.7772                   0.7772
     b16 first 8 PCA    0.6360         0.6360              0.6303                   0.6303
   effb4       PLS 8    0.6924         0.6924              0.5796                   0.5796
   effb4 first 8 PCA    0.5219         0.5219              0.5081                   0.5081


## 9. The context table

Per backbone: each row's 8 inputs under the three models, then the probe on all features. Val with its bootstrap interval, the held-out mean and the worst held-out manipulation. The twin column is what the additive circuit can reach on those inputs; the real-circuit numbers come from the final-circuit notebook.

In [9]:
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
order = {"twin": 0, "logistic regression": 1, "MLP": 2, "linear probe": 3}
for b in backbones:
    t = table[table["backbone"] == b].copy()
    t["m"] = t["model"].map(order)
    t = t.sort_values(["source", "row", "m"])
    print(f"\n{b}")
    print(t[["row", "model", "weights", "val", "val low", "val high", "held-out mean", "held-out worst"]]
          .round(3).to_string(index=False))

wide = []
for (b, row), g in table[table["model"] != "linear probe"].groupby(["backbone", "row"], sort=False):
    r = {"backbone": b, "row": row}
    for kind in ["twin", "logistic regression", "MLP"]:
        x = g[g["model"] == kind]
        r[kind + " val"] = x["val"].iloc[0]
        r[kind + " held-out"] = x["held-out mean"].iloc[0]
    r["MLP minus twin (val)"] = r["MLP val"] - r["twin val"]
    wide.append(r)
wide = pd.DataFrame(wide)
wide.to_csv(f"{out_dir}/context_wide.csv", index=False)
print()
print(wide.round(3).to_string(index=False))
print("\nlargest MLP gain over the twin on val: %.3f" % wide["MLP minus twin (val)"].max())


l14
              row               model  weights   val  val low  val high  held-out mean  held-out worst
GA + held-out top                twin        9 0.879    0.850     0.907          0.811           0.570
GA + held-out top logistic regression        9 0.881    0.852     0.908          0.814           0.567
GA + held-out top                 MLP       41 0.880    0.850     0.907          0.809           0.554
         MI top 8                twin       17 0.849    0.817     0.877          0.801           0.549
         MI top 8 logistic regression        9 0.852    0.821     0.880          0.802           0.546
         MI top 8                 MLP       41 0.848    0.815     0.877          0.800           0.542
            PLS 8                twin        9 0.888    0.860     0.915          0.813           0.574
            PLS 8 logistic regression        9 0.891    0.864     0.917          0.815           0.570
            PLS 8                 MLP       41 0.889    0.861     0.

## 10. Run log

In [10]:
info = {"finished": time.strftime("%Y-%m-%d %H:%M:%S"), "machine": machine_name, "backbones": backbones,
        "methods": methods, "use_ablation": use_ablation and ablation_log is not None, "use_finalists": use_finalists,
        "c_grid": c_grid, "mlp_hidden": mlp_hidden, "alpha_grid": alpha_grid, "n_pls": n_pls, "n_boot": n_boot,
        "n_jobs": n_jobs, "total_minutes": round((time.time() - start_all) / 60, 1),
        "python": platform.python_version(), "numpy": np.__version__, "sklearn": sklearn.__version__}
with open(f"{out_dir}/runs.jsonl", "a") as f:
    f.write(json.dumps(info) + "\n")
!ls -lh {out_dir}

total 52K
-rw-r--r-- 1 root root  39K Nov 26 18:39 context_table.csv
-rw-r--r-- 1 root root 5.1K Nov 26 18:39 context_wide.csv
-rw-r--r-- 1 root root  424 Nov 26 18:39 runs.jsonl
